## Assignment 34: Text summarization using Langchain

# Part 1: Load and Prepare Text

In [1]:
from dotenv import load_dotenv

In [8]:
load_dotenv()

True

In [2]:

from pathlib import Path


In [4]:

file_path = Path("article.txt")

In [5]:
text = file_path.read_text(encoding='utf-8')

In [6]:
print(len(text))

10298


In [7]:
text[:1000]

'Artificial intelligence has become an important part of modern software engineering.\nThe development of large language models has changed the way developers write,\ntest, document, and maintain software applications. Earlier software development\nmainly depended on manually written source code, predefined rules, and traditional\ndevelopment tools. Modern AI systems can assist developers with many of these\nactivities while still requiring human supervision.\n\nOne of the most common applications of artificial intelligence in software\nengineering is code generation. Large language models can generate functions,\nclasses, configuration files, database queries, test cases, and documentation from\nnatural language descriptions. This allows developers to describe the desired\nbehavior of a component and receive an initial implementation. However, generated\ncode should not automatically be considered correct. Developers still need to\nreview the code, understand its behavior, test it, an

## Task 2: Prompt Based Summarization

In [10]:
import os
from langchain_openai import ChatOpenAI

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [12]:
from langchain_core.prompts import PromptTemplate

In [13]:
summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert document summarizer.

Summarize the following document accurately.

Requirements:
- Preserve the main ideas.
- Remove unnecessary repetition.
- Do not invent information.
- Keep important technical details.
- Write a concise and meaningful summary.

Document:
{text}

Summary:
"""
)

In [14]:
chain = summary_prompt | llm

response = chain.invoke({"text": text})

summary = response.content


In [15]:
summary

"Artificial intelligence (AI) has significantly transformed software engineering, particularly through the use of large language models that assist developers in writing, testing, documenting, and maintaining software applications. Key applications of AI include code generation, where developers can describe desired functionalities and receive initial implementations, though generated code requires thorough review and testing. AI tools also enhance productivity by automating repetitive tasks, such as generating test cases and explaining code.\n\nIn software testing, AI can analyze source code to generate potential test cases and identify edge cases, but these tests must be validated by humans. AI aids in debugging by analyzing logs and suggesting fixes, although the quality of suggestions depends on the context provided. Additionally, AI can help generate and update documentation, but it must be regularly reviewed to ensure accuracy.\n\nAI's integration into software development raises

## Task 3: Prompt Variations

In [17]:
short_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert summarizer.

Summarize the following document in exactly 5–6 concise lines.

Rules:
- Include only the most important information.
- Preserve the main message.
- Do not add information that is not present.
- Avoid unnecessary details.

Document:
{text}

5–6 Line Summary:
"""
)


In [18]:
short_chain = short_summary_prompt | llm

short_response = short_chain.invoke({
    "text": text
})


In [ ]:
bullet_summary_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert document summarizer.

Summarize the following document using clear bullet points.

Requirements:
- Identify the major ideas.
- Include important technical concepts.
- Keep each bullet concise.
- Do not invent information.
- Use approximately 8–12 bullet points.

Document:
{text}

Bullet-Point Summary:
"""
)



===== BULLET-POINT SUMMARY =====


In [24]:
bullet_chain = bullet_summary_prompt | llm

bullet_response = bullet_chain.invoke({
    "text": text
})

In [ ]:
bullet_response.content

In [22]:
short_response.content

'Artificial intelligence is transforming software engineering by enhancing code generation, testing, debugging, and documentation processes. AI tools can improve developer productivity by automating repetitive tasks, but human oversight remains essential to ensure code quality and security. The integration of AI requires careful consideration of organizational policies, validation mechanisms, and observability metrics. As AI becomes a collaborative tool, developers will focus more on system design and validation while leveraging AI for implementation support. Ultimately, AI should be viewed as a tool that complements human expertise rather than a replacement.'

comparison

In [ ]:
print(short_response.content)

In [ ]:
print(bullet_response.content)

# PART 2: Stuff Summairzation Chain

## Task 4: Why Stuff Chain is needed 

1. What is a Stuff Summarization Chain?
- A Stuff chain is a summarization strategy where all input documents are inserted into a single prompt and sent to the LLM in one call.

2. When Is Stuff Suitable?
- The document is relatively small.
- All input text fits inside the model's context window.
- A single-pass summary is sufficient.
- You want to preserve relationships between different parts of the document.
- You want a simple implementation.

3. Limitations of Stuff Chain
- The biggest limitation is context-window size.

## Task 5: Implement Stuff Summarization Chain

In [25]:
from langchain_classic.chains.summarize import load_summarize_chain

c:\Users\arunk\anaconda3\envs\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
from langchain_core.documents import Document

document = Document(page_content=text)

docs = [document]

In [27]:
stuff_chain = load_summarize_chain(llm,chain_type="stuff")

In [28]:
stuff_summary = stuff_chain.invoke(docs)

In [29]:
stuff_summary["output_text"]

"Artificial intelligence (AI) has significantly transformed software engineering, particularly through the use of large language models that assist in code generation, testing, debugging, and documentation. While AI can enhance developer productivity by automating repetitive tasks and generating initial code implementations, human oversight remains essential to ensure correctness and adherence to requirements. AI tools can also improve software testing by generating test cases and identifying edge cases, but these tests require validation.\n\nAI's integration into software development raises concerns about security and privacy, necessitating careful management of sensitive information and adherence to organizational policies. The role of software engineers is evolving; they now focus more on reviewing AI-generated code and system design rather than routine coding tasks. Additionally, AI impacts software architecture, requiring developers to consider factors like latency and security wh

## Task 6: Comparison

In [ ]:
prompt_chain = summary_prompt | llm

prompt_result = prompt_chain.invoke({
    "text": text
})

In [ ]:
stuff_chain = load_summarize_chain(
    llm,
    chain_type="stuff"
)

stuff_result = stuff_chain.invoke(docs)

In [ ]:
print(prompt_result.content)

In [ ]:
print(stuff_result["output_text"])

Common in Both Prompt Summary and Stuff chain
1. Both are good for small documents
2. bad for huge documents
3. context window limitation
4. prompt customization is very high
5. both use langchain

# Part 3: Map Reduce Summarization

## Task 7: Why Map Reduce is Needed

1. Why do large documents need Map-Reduce?

The biggest problem with Stuff is that it sends the entire document to the LLM in one prompt.

2. How do Map and Reduce steps work?
- Map step: Each chunk is summarized independently.
- Reduce step: The individual summaries are combined and summarized again.
- Main advantage: The original huge document does not need to fit into a single LLM prompt.

## Task 8: Implement Map-Reduce

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(text)

In [ ]:
len(chunks)

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=chunk,
        metadata={"chunk": i + 1}
    )
    for i, chunk in enumerate(chunks)
]

In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

In [ ]:


map_reduce_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce"
)

In [ ]:
result = map_reduce_chain.invoke(docs)

In [ ]:
result["output_text"]

## Task 9: Analyse Map Output (Optional)

## Task 10: Understanding Refine Chain

1. How does Refine summarization work?

- Refine works sequentially.
- Instead of independently summarizing every chunk and then combining the results, Refine creates an initial summary and then repeatedly improves it using subsequent chunks.

2. Map-Reduce vs Refine
- map reduce processing parallel style while refine process sequential
- map reduce summarize every chunk, while refine process summarize first chunk

## Task 11: Implement Refine Chain

In [ ]:
refine_chain = load_summarize_chain(llm,chain_type="refine")

In [ ]:
refine_result = refine_chain.invoke(docs)

In [ ]:
print(refine_result["output_text"])

## Task 12: Comparison of All Summarization Methods

You now have four methods:

1. Prompt-based
2. Stuff
3. Map-Reduce
4. Refine

In [ ]:
prompt_result = prompt_chain.invoke({
    "text": text
})

prompt_summary = prompt_result.content
prompt_summary

In [ ]:
stuff_result = stuff_chain.invoke(docs)

stuff_summary = stuff_result["output_text"]
stuff_summary

In [ ]:
map_reduce_result = map_reduce_chain.invoke(docs)

map_reduce_summary = map_reduce_result["output_text"]
map_reduce_summary

In [ ]:
refine_result = refine_chain.invoke(docs)

refine_summary = refine_result["output_text"]

refine_summary

| Method     | Main Strength                       | Main Limitation                         |
| ---------- | ----------------------------------- | --------------------------------------- |
| Prompt     | Maximum prompt control              | Context-window limitation               |
| Stuff      | Very simple document-chain approach | Entire input must fit context           |
| Map-Reduce | Handles large documents by chunking | Intermediate summaries can lose details |
| Refine     | Maintains an evolving summary       | Sequential and potentially slower       |


## Task 13: Mini Project: Document Summarizer

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.summarize import load_summarize_chain

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [ ]:
prompt_template = PromptTemplate(
    input_variables=["text"],
    template="""
You are an expert document summarizer.

Summarize the following document accurately.

Requirements:
- Preserve the main ideas.
- Preserve important details.
- Remove unnecessary repetition.
- Do not invent information.
- Keep the summary concise and meaningful.

Document:
{text}

Summary:
"""
)

In [ ]:

prompt_chain = prompt_template | llm

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

In [ ]:
def summarize_document(
    text: str,
    method: str = "map_reduce"
) -> str:

    if not text or not text.strip():
        raise ValueError("Text cannot be empty.")

    method = method.lower().strip()


    if method == "prompt":

        response = prompt_chain.invoke({
            "text": text
        })

        return response.content

    chunks = text_splitter.split_text(text)

    docs = [
        Document(
            page_content=chunk,
            metadata={
                "chunk": i + 1
            }
        )
        for i, chunk in enumerate(chunks)
    ]

    if method == "stuff":

        chain = load_summarize_chain(
            llm,
            chain_type="stuff"
        )

        result = chain.invoke(docs)

        return result["output_text"]

    elif method == "map_reduce":

        chain = load_summarize_chain(
            llm,
            chain_type="map_reduce"
        )

        result = chain.invoke(docs)

        return result["output_text"]

    elif method == "refine":

        chain = load_summarize_chain(
            llm,
            chain_type="refine"
        )

        result = chain.invoke(docs)

        return result["output_text"]

    else:

        raise ValueError(
            "Invalid method. Choose from: "
            "'prompt', 'stuff', 'map_reduce', 'refine'."
        )

In [31]:
text = Path("article.txt").read_text(encoding="utf-8")

In [ ]:
summarize_document(text,method="prompt")

In [ ]:
summarize_document(text,method="stuff")

In [ ]:
summarize_document(text,method="map_reduce")

In [ ]:
summarize_document(text,method="refine")

## Task 13: Observations and Insights

1. Best Summarization strategy for very long documents
- There is no universally best strategy; for very long documents, Map-Reduce is generally the most practical of the methods implemented in this assignment.

2. Trade-offs between speed and quality
- Prompt-Based: Simple and fast.
- Stuff: Scales better for large documents.
- Map-Reduce: Each new chunk can update the existing summary.
- Refine: Each new chunk can update the existing summary.

3. Real world use cases of each method
- Prompt-Based Summarization: Best suited to relatively short documents. eg. email summarization, short articles
- Stuff chain: Useful when the complete document comfortably fits within the model's context window. eg. short researcha paper
- Map-Reduce Chain: Useful for large documents where independent sections can be summarized separately. eg. books, long research paper.
- Refine Chain: Useful when the order of information matters and the summary should evolve as additional sections are processed. eg. long sequential meeting transcripts
